# 03 — MAP Fitting

Tests `fitting.py` on synthetic data generated by each model.

**Goals:**
1. Verify `fit_map` recovers true parameters (close agreement for key params)
2. Verify BIC/AIC selects the correct generating model when all models are fit to RRSF data

**`pi0_init` note:** In real analysis (notebook 05), `pi0_init` is computed once from the
group-level histogram of participants' first actions using `fitting.compute_pi0_init`,
then passed unchanged to every `fit_map` call. Here we use `DEFAULT_PI0` (uniform) as a
stand-in — sufficient for verifying the fitting machinery.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from env import Env
from models import ALL_MODELS
from models.base import DEFAULT_PI0
from fitting import fit_map, bic, aic, compute_pi0_init, log_prior

# ── Shared setup ──────────────────────────────────────────────────────────────
TRIALS  = Env.make_trial_sequence(seed=0)
N_OBS   = len(TRIALS) * 2          # 160 — two actions per trial
PI0     = DEFAULT_PI0               # uniform stand-in for pi0_init

# All models except RRMB (excluded: computationally expensive, paper §Methods)
FIT_MODELS = {k: v for k, v in ALL_MODELS.items() if k != 'RRMB'}

# True parameters used for synthetic data generation
TRUE_PARAMS = {
    'MF':   {'gamma': 0.9, 'alpha_Q': 0.3,   'lam': 1.0},
    'SF':   {'gamma': 0.9, 'alpha_psi': 0.3,  'lam': 1.0},
    'MB':   {'gamma': 0.9, 'lam': 1.0,        'alpha_t': 0.3, 'alpha_phi': 0.3},
    'MFP':  {'gamma': 0.9, 'alpha_Q': 0.3,   'lam': 1.0, 'alpha_0': 0.3, 'h': 1.0, 'epsilon': 0.05},
    'SFP':  {'gamma': 0.9, 'alpha_psi': 0.3,  'lam': 1.0, 'alpha_0': 0.3, 'h': 1.0, 'epsilon': 0.05},
    'RRMF': {'lam': 2.0,   'gamma': 0.9,      'alpha': 0.3, 'alpha_0': 0.1, 'epsilon': 0.05},
    'RRSF': {'lam': 2.0,   'gamma': 0.9,      'alpha_psi': 0.3, 'alpha_0': 0.1, 'epsilon': 0.05},
}

print('Setup complete.')
print(f'N_OBS = {N_OBS}  (= {len(TRIALS)} trials × 2 actions)')

## Prior check

Verify that `log_prior` returns 0 for [0,1] params and the HalfNormal(20) value for (0,∞) params.

In [ ]:
from scipy.stats import halfnorm

# RRSF has both types: lam (log → HalfNorm) and the rest (logit → Uniform)
mod  = ALL_MODELS['RRSF']
spec = mod.PARAM_SPEC

params_test = {'lam': 2.0, 'gamma': 0.9, 'alpha_psi': 0.3, 'alpha_0': 0.1, 'epsilon': 0.05}
lp = log_prior(params_test, spec)
expected = halfnorm.logpdf(2.0, scale=20.0)   # only lam contributes

assert np.isclose(lp, expected), f'log_prior mismatch: {lp} vs {expected}'
print(f'log_prior test passed ✓  (log_prior = {lp:.4f}, HalfNorm(λ=2|σ=20) = {expected:.4f})')

## Sanity check: fit MF on synthetic data

Fit the simplest model (3 params) as a quick end-to-end test before running all models.

In [ ]:
mf_mod    = ALL_MODELS['MF']
mf_true   = TRUE_PARAMS['MF']
mf_rng    = np.random.default_rng(0)

mf_actions = mf_mod.simulate(TRIALS, mf_true, PI0, mf_rng)
mf_params, mf_ll = fit_map(mf_mod, mf_actions, TRIALS, PI0, n_restarts=40, seed=0)

print('MF parameter recovery (40 restarts):')
print(f"  {'param':<12}  {'true':>8}  {'recovered':>10}  {'error':>8}")
for k, v_true in mf_true.items():
    v_rec = mf_params[k]
    print(f"  {k:<12}  {v_true:8.3f}  {v_rec:10.3f}  {abs(v_rec - v_true):8.3f}")

mf_bic = bic(mf_ll, mf_mod.N_PARAMS, N_OBS)
mf_aic = aic(mf_ll, mf_mod.N_PARAMS)
print(f'\n  LL = {mf_ll:.2f}  |  BIC = {mf_bic:.2f}  |  AIC = {mf_aic:.2f}')

## Fit all models to their own synthetic data

Each model generates one synthetic participant (80 trials), then `fit_map` is run with 40 restarts.
RRMB is excluded (computationally prohibitive; also excluded from paper's model recovery).

In [ ]:
fit_results = {}

for name, mod in FIT_MODELS.items():
    true_p  = TRUE_PARAMS[name]
    rng     = np.random.default_rng(42)
    actions = mod.simulate(TRIALS, true_p, PI0, rng)

    rec_p, rec_ll = fit_map(mod, actions, TRIALS, PI0, n_restarts=40, seed=0)

    n_p     = mod.N_PARAMS
    rec_bic = bic(rec_ll, n_p, N_OBS)
    rec_aic = aic(rec_ll, n_p)

    fit_results[name] = {
        'true': true_p, 'recovered': rec_p,
        'll': rec_ll, 'bic': rec_bic, 'aic': rec_aic,
    }
    print(f'{name:6s}  LL={rec_ll:7.2f}  BIC={rec_bic:7.2f}  AIC={rec_aic:7.2f}')

print('\nDone.')

## True vs. recovered parameters

In [ ]:
colors = px.colors.qualitative.Plotly

for name, res in fit_results.items():
    true_p = res['true']
    rec_p  = res['recovered']
    params = list(true_p.keys())

    fig = go.Figure([
        go.Bar(name='True',      x=params, y=[true_p[k] for k in params],
               marker_color=colors[0], opacity=0.8),
        go.Bar(name='Recovered', x=params, y=[rec_p[k]  for k in params],
               marker_color=colors[1], opacity=0.8),
    ])
    fig.update_layout(
        title=f'{name} — true vs. recovered parameters',
        barmode='group', width=600, height=300,
        yaxis_title='Parameter value', legend_title='',
        margin=dict(t=50),
    )
    fig.show()

## Cross-model BIC: does RRSF win on RRSF-generated data?

Simulate one RRSF participant, fit all 7 models (excl. RRMB), compare BIC.
The correct model should achieve the lowest (best) BIC.

In [ ]:
# Simulate RRSF participant
rrsf_true    = TRUE_PARAMS['RRSF']
rrsf_actions = ALL_MODELS['RRSF'].simulate(TRIALS, rrsf_true, PI0, np.random.default_rng(7))

# Fit all 7 models to the same RRSF-generated data
cross_bic = {}
cross_aic = {}

for name, mod in FIT_MODELS.items():
    rec_p, rec_ll = fit_map(mod, rrsf_actions, TRIALS, PI0, n_restarts=40, seed=0)
    cross_bic[name] = bic(rec_ll, mod.N_PARAMS, N_OBS)
    cross_aic[name] = aic(rec_ll, mod.N_PARAMS)
    print(f'{name:6s}  LL={rec_ll:7.2f}  BIC={cross_bic[name]:7.2f}  AIC={cross_aic[name]:7.2f}')

best_bic = min(cross_bic, key=cross_bic.get)
best_aic = min(cross_aic, key=cross_aic.get)
print(f'\nBest by BIC: {best_bic}  |  Best by AIC: {best_aic}')
print('Expected: RRSF wins (or close competitor)')

In [ ]:
# Plot BIC and AIC for cross-model comparison
names  = list(cross_bic.keys())
# Relative to RRSF: ΔBIC = BIC_model − BIC_RRSF  (positive = worse than RRSF)
rrsf_bic_val = cross_bic['RRSF']
rrsf_aic_val = cross_aic['RRSF']

delta_bic = [cross_bic[n] - rrsf_bic_val for n in names]
delta_aic = [cross_aic[n] - rrsf_aic_val for n in names]

fig = go.Figure([
    go.Bar(name='ΔBIC', x=names, y=delta_bic, marker_color=colors[0], opacity=0.85),
    go.Bar(name='ΔAIC', x=names, y=delta_aic, marker_color=colors[1], opacity=0.85),
])
fig.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='RRSF baseline')
fig.update_layout(
    title='Cross-model comparison on RRSF-simulated data (ΔBIC / ΔAIC vs RRSF)',
    barmode='group', width=700, height=380,
    yaxis_title='Δ score (lower = better; 0 = RRSF)',
    xaxis_title='Model fitted to RRSF data',
)
fig.show()

## compute_pi0_init: group-level first-action histogram

Demonstrates how to build `pi0_init` from a group of participants.
In the real analysis (notebook 05), this is called once on all participants' data
before any individual fitting.

In [ ]:
# Simulate a small group of 20 RRSF participants to demonstrate pi0_init computation
N_GROUP = 20
group_rng = np.random.default_rng(99)
group_actions = []

for i in range(N_GROUP):
    # Slightly perturb params across participants
    p = {
        'lam':       float(group_rng.uniform(1.0, 4.0)),
        'gamma':     0.9,
        'alpha_psi': float(group_rng.uniform(0.1, 0.5)),
        'alpha_0':   float(group_rng.uniform(0.05, 0.3)),
        'epsilon':   0.05,
    }
    acts = ALL_MODELS['RRSF'].simulate(TRIALS, p, PI0, np.random.default_rng(i))
    group_actions.append(acts)

# Collect all actions per state across all trials for each participant
first_actions = {s: [] for s in Env.NON_TERMINAL}
for acts in group_actions:
    for a0, a1 in acts:                  # loop over all 80 trials
        first_actions[0].append(a0)
        s1 = Env.step(0, a0)             # intermediate state actually visited
        first_actions[s1].append(a1)

pi0_group = compute_pi0_init(first_actions)

print('Group-level pi0_init (from 20 synthetic RRSF participants, all trials):')
for s, dist in pi0_group.items():
    print(f'  s={s}: {np.round(dist, 3)}  (sums to {dist.sum():.3f})')

print('\nNote: In real analysis, use all ~126 participants with this same collection loop.')